In [1]:
import duckdb
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error

BASE = "C:/Users/HP/Downloads/DS/da9/walmart-demand-forecast"
PROCESSED = f"{BASE}/data/processed"
con = duckdb.connect()

# Lấy danh sách Nhánh B + C
streak_df = pd.read_csv(f"{PROCESSED}/gap_classification.csv")
weeks_count = con.execute(f"""
    SELECT Store, Dept, COUNT(DISTINCT Date) AS n_weeks
    FROM read_parquet('{PROCESSED}/walmart_clean.parquet') GROUP BY Store, Dept
""").df()
merged = weeks_count.merge(streak_df[['Store','Dept','max_streak']], on=['Store','Dept'], how='left')
merged['max_streak'] = merged['max_streak'].fillna(0)

nhanh_bc = merged[~((merged['n_weeks'] >= 104) & (merged['max_streak'] <= 10))][['Store','Dept']]
print(f"Số cặp Nhánh B+C: {len(nhanh_bc)}")

# Chạy SQL feature engineering 

with open(f"{BASE}/sql/02_features_nhanh_bc.sql", "r", encoding="utf-8") as f:
    sql_query = f.read().format(PROCESSED=PROCESSED)

con.execute(sql_query)

features_df = con.execute("""
    SELECT * FROM nhanh_bc_features
""").df()

# Chỉ giữ đúng các cặp thuộc Nhánh B+C
features_df = features_df.merge(nhanh_bc, on=['Store','Dept'])
features_df['Date'] = pd.to_datetime(features_df['Date'])
features_df = features_df.sort_values(['Store','Dept','Date'])

# Encode Store_Type
features_df['Store_Type'] = features_df['Store_Type'].astype('category')

feature_cols = ['is_active_season', 'lag_1', 'lag_52', 'rolling_mean_4',
                 'pct_active_recent_8w', 'IsHoliday', 'Temperature', 'Fuel_Price',
                 'CPI', 'Unemployment', 'Store_Type', 'Store_Size']

# Train/test split theo thời gian - giữ 8 tuần cuối cho MỌI cặp làm test
cutoff_date = features_df['Date'].max() - pd.Timedelta(weeks=8)
train_df = features_df[features_df['Date'] <= cutoff_date].dropna(subset=['lag_1','lag_52'])
test_df = features_df[features_df['Date'] > cutoff_date]

print(f"Train: {len(train_df)} dòng, Test: {len(test_df)} dòng")

model = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    num_leaves=31, random_state=42, verbose=-1
)
model.fit(train_df[feature_cols], train_df['Weekly_Sales'],
          categorical_feature=['Store_Type'])

test_df = test_df.dropna(subset=['lag_1','lag_52']).copy()
test_df['predicted'] = model.predict(test_df[feature_cols])

# Đánh giá riêng: tuần đang active vs tuần off-season - KHÔNG gộp chung
# (vì MAPE trên tuần off-season=0 sẽ vô nghĩa toán học, giống bài học MAPE trước đó)
active_test = test_df[test_df['is_active_season'] == 1]
mape_active = mean_absolute_percentage_error(active_test['Weekly_Sales'], active_test['predicted']) * 100
print(f"\nMAPE trên các tuần ĐANG ACTIVE: {mape_active:.2f}%")

# Đánh giá riêng khả năng model dự đoán ĐÚNG trạng thái active/off-season
off_season_test = test_df[test_df['is_active_season'] == 0]
if len(off_season_test) > 0:
    avg_predicted_offseason = off_season_test['predicted'].mean()
    print(f"Dự báo trung bình cho tuần OFF-SEASON (nên gần 0): {avg_predicted_offseason:,.2f}")

feature_importance = pd.DataFrame({
    'feature': feature_cols, 'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print("\n=== FEATURE IMPORTANCE ===")
print(feature_importance)

Số cặp Nhánh B+C: 484
Train: 40172 dòng, Test: 3872 dòng

MAPE trên các tuần ĐANG ACTIVE: 7308773457604855.00%
Dự báo trung bình cho tuần OFF-SEASON (nên gần 0): -0.03

=== FEATURE IMPORTANCE ===
                 feature  importance
2                 lag_52        1322
1                  lag_1        1223
3         rolling_mean_4         981
7             Fuel_Price         655
6            Temperature         652
8                    CPI         651
9           Unemployment         287
11            Store_Size         174
4   pct_active_recent_8w          97
5              IsHoliday          83
10            Store_Type          68
0       is_active_season           0


In [2]:
# Sửa cách đánh giá - dùng WAPE (Weighted Absolute Percentage Error) để tránh bùng nổ khi actual gần 0 (đúng bài học từ Nhánh A)

def wape(actual, predicted):
    return np.sum(np.abs(actual - predicted)) / np.sum(np.abs(actual)) * 100

wape_active = wape(active_test['Weekly_Sales'], active_test['predicted'])
print(f"WAPE trên các tuần ĐANG ACTIVE: {wape_active:.2f}%")

# So sánh với naive baseline: dự báo = lag_52 (cùng kỳ năm trước)
wape_naive = wape(active_test['Weekly_Sales'], active_test['lag_52'])
print(f"WAPE naive (dùng lag_52 làm baseline): {wape_naive:.2f}%")
print(f"LightGBM {'THẮNG' if wape_active < wape_naive else 'THUA'} naive")

# Kiểm tra thêm: bao nhiêu case actual gần 0 gây vấn đề MAPE (để hiểu rõ, không phải sửa)
near_zero = active_test[active_test['Weekly_Sales'].abs() < 10]
print(f"\nSố dòng active có |Weekly_Sales| < 10 USD: {len(near_zero)} ({len(near_zero)/len(active_test)*100:.2f}%)")

# Đánh giá theo quartile doanh số - kỳ vọng lặp lại pattern giống Nhánh A
active_test_copy = active_test.copy()
active_test_copy['abs_error'] = (active_test_copy['Weekly_Sales'] - active_test_copy['predicted']).abs()
avg_sales_per_pair = active_test_copy.groupby(['Store','Dept'])['Weekly_Sales'].transform('mean')
active_test_copy['sales_quartile'] = pd.qcut(avg_sales_per_pair, 4, labels=['Q1_nhỏ','Q2','Q3','Q4_lớn'], duplicates='drop')

wape_by_quartile = active_test_copy.groupby('sales_quartile', observed=True).apply(
    lambda g: wape(g['Weekly_Sales'], g['predicted'])
)
print("\n=== WAPE THEO QUARTILE DOANH SỐ ===")
print(wape_by_quartile)

WAPE trên các tuần ĐANG ACTIVE: 32.33%
WAPE naive (dùng lag_52 làm baseline): 30.96%
LightGBM THUA naive

Số dòng active có |Weekly_Sales| < 10 USD: 213 (19.65%)

=== WAPE THEO QUARTILE DOANH SỐ ===
sales_quartile
Q1_nhỏ    317.588128
Q2        120.333292
Q3         92.239127
Q4_lớn     25.996674
dtype: float64


C:\Users\HP\AppData\Local\Temp\ipykernel_9236\774864597.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wape_by_quartile = active_test_copy.groupby('sales_quartile', observed=True).apply(


In [3]:
#thêm Store, Dept làm categorical feature
features_df['Store'] = features_df['Store'].astype('category')
features_df['Dept'] = features_df['Dept'].astype('category')

feature_cols_v2 = ['Store', 'Dept', 'is_active_season', 'lag_1', 'lag_52', 'rolling_mean_4',
                    'pct_active_recent_8w', 'IsHoliday', 'Temperature', 'Fuel_Price',
                    'CPI', 'Unemployment', 'Store_Type', 'Store_Size']

train_df_v2 = features_df[features_df['Date'] <= cutoff_date].dropna(subset=['lag_1','lag_52'])
test_df_v2 = features_df[features_df['Date'] > cutoff_date].dropna(subset=['lag_1','lag_52']).copy()

model_v2 = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    num_leaves=31, random_state=42, verbose=-1
)
model_v2.fit(train_df_v2[feature_cols_v2], train_df_v2['Weekly_Sales'],
             categorical_feature=['Store', 'Dept', 'Store_Type'])

test_df_v2['predicted'] = model_v2.predict(test_df_v2[feature_cols_v2])

active_test_v2 = test_df_v2[test_df_v2['is_active_season'] == 1]
wape_v2 = wape(active_test_v2['Weekly_Sales'], active_test_v2['predicted'])
wape_naive_v2 = wape(active_test_v2['Weekly_Sales'], active_test_v2['lag_52'])
print(f"WAPE (có Store+Dept feature): {wape_v2:.2f}%")
print(f"WAPE naive: {wape_naive_v2:.2f}%")
print(f"{'THẮNG' if wape_v2 < wape_naive_v2 else 'THUA'} naive")

active_test_v2_copy = active_test_v2.copy()
avg_sales_per_pair_v2 = active_test_v2_copy.groupby(['Store','Dept'])['Weekly_Sales'].transform('mean')
active_test_v2_copy['sales_quartile'] = pd.qcut(avg_sales_per_pair_v2, 4, labels=['Q1_nhỏ','Q2','Q3','Q4_lớn'], duplicates='drop')
wape_by_quartile_v2 = active_test_v2_copy.groupby('sales_quartile', observed=True).apply(
    lambda g: wape(g['Weekly_Sales'], g['predicted'])
)
print("\n=== WAPE THEO QUARTILE (sau khi thêm Store+Dept) ===")
print(wape_by_quartile_v2)

fi_v2 = pd.DataFrame({'feature': feature_cols_v2, 'importance': model_v2.feature_importances_}).sort_values('importance', ascending=False)
print("\n=== FEATURE IMPORTANCE (v2) ===")
print(fi_v2)

WAPE (có Store+Dept feature): 32.09%
WAPE naive: 30.96%
THUA naive

=== WAPE THEO QUARTILE (sau khi thêm Store+Dept) ===
sales_quartile
Q1_nhỏ    233.593658
Q2        119.728274
Q3        101.736773
Q4_lớn     25.746983
dtype: float64

=== FEATURE IMPORTANCE (v2) ===
                 feature  importance
3                  lag_1        1272
4                 lag_52        1270
5         rolling_mean_4         893
8            Temperature         650
9             Fuel_Price         600
10                   CPI         588
11          Unemployment         252
1                   Dept         144
13            Store_Size         108
6   pct_active_recent_8w          92
7              IsHoliday          73
12            Store_Type          70
0                  Store          69
2       is_active_season           0


C:\Users\HP\AppData\Local\Temp\ipykernel_9236\3942864334.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  avg_sales_per_pair_v2 = active_test_v2_copy.groupby(['Store','Dept'])['Weekly_Sales'].transform('mean')
C:\Users\HP\AppData\Local\Temp\ipykernel_9236\3942864334.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wape_by_quartile_v2 = active_test_v2_copy.groupby('sales_quartile', observed=True).apply(


In [4]:
#so sánh theo từng quartile
wape_naive_by_quartile = active_test_v2_copy.groupby('sales_quartile', observed=True).apply(
    lambda g: wape(g['Weekly_Sales'], g['lag_52'])
)
print("=== WAPE NAIVE THEO QUARTILE (để so sánh công bằng) ===")
print(wape_naive_by_quartile)

compare_df = pd.DataFrame({
    'LightGBM': wape_by_quartile_v2,
    'Naive': wape_naive_by_quartile
})
compare_df['LightGBM_thắng'] = compare_df['LightGBM'] < compare_df['Naive']
print("\n=== SO SÁNH TRỰC TIẾP ===")
print(compare_df)

=== WAPE NAIVE THEO QUARTILE (để so sánh công bằng) ===
sales_quartile
Q1_nhỏ    108.366808
Q2         83.273147
Q3         76.553508
Q4_lớn     27.216672
dtype: float64

=== SO SÁNH TRỰC TIẾP ===
                  LightGBM       Naive  LightGBM_thắng
sales_quartile                                        
Q1_nhỏ          233.593658  108.366808           False
Q2              119.728274   83.273147           False
Q3              101.736773   76.553508           False
Q4_lớn           25.746983   27.216672            True


C:\Users\HP\AppData\Local\Temp\ipykernel_9236\3583047550.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wape_naive_by_quartile = active_test_v2_copy.groupby('sales_quartile', observed=True).apply(


In [5]:
#kiểm tra cỡ mẫu
n_per_quartile = active_test_v2_copy.groupby('sales_quartile', observed=True).size()
n_pairs_per_quartile = active_test_v2_copy.groupby('sales_quartile', observed=True)[['Store','Dept']].apply(
    lambda g: g.drop_duplicates().shape[0]
)
print("=== SỐ DÒNG TEST VÀ SỐ CẶP STORE-DEPT MỖI QUARTILE ===")
print(pd.DataFrame({'n_rows': n_per_quartile, 'n_pairs': n_pairs_per_quartile}))

=== SỐ DÒNG TEST VÀ SỐ CẶP STORE-DEPT MỖI QUARTILE ===
                n_rows  n_pairs
sales_quartile                 
Q1_nhỏ             275       87
Q2                 267       53
Q3                 274       45
Q4_lớn             268       38


In [4]:
#kiểm tra lần cuối
import pandas as pd

PROCESSED = f"{BASE}/data/processed"
forecast_bc = pd.read_csv(f"{PROCESSED}/production_forecast_nhanh_bc.csv")

print("=== KIỂM TRA NHANH FORECAST NHÁNH B+C ===")
print(f"Tổng dòng: {len(forecast_bc)}")
print(f"Forecast âm: {(forecast_bc['forecast'] < 0).sum()}")
print(forecast_bc.groupby('method')['forecast'].describe())

# Kiểm tra Q1-Q3 (seasonal naive) - có bao nhiêu dòng forecast = 0
naive_zero = forecast_bc[(forecast_bc['method']=='Seasonal_Naive') & (forecast_bc['forecast']==0)]
print(f"\nSố dòng Seasonal_Naive forecast = 0 (đang off-season): {len(naive_zero)} ({len(naive_zero)/len(forecast_bc[forecast_bc['method']=='Seasonal_Naive'])*100:.1f}%)")

=== KIỂM TRA NHANH FORECAST NHÁNH B+C ===
Tổng dòng: 1936
Forecast âm: 0
                 count         mean          std  min  25%        50%  \
method                                                                  
LightGBM         484.0  1445.833964  4937.441418  0.0  0.0  48.935097   
Seasonal_Naive  1452.0     8.238698    31.248644  0.0  0.0   0.000000   

                       75%           max  
method                                    
LightGBM        708.013265  40300.545276  
Seasonal_Naive    0.000000    649.000000  

Số dòng Seasonal_Naive forecast = 0 (đang off-season): 1171 (80.6%)


In [5]:
#gộp
forecast_a = pd.read_csv(f"{PROCESSED}/production_forecast_nhanh_a_final.csv")
forecast_a_slim = forecast_a[['Store','Dept','Date','forecast','lower_95','upper_95','method']]

forecast_bc_slim = forecast_bc[['Store','Dept','Date','forecast','method']].copy()
forecast_bc_slim['lower_95'] = None 
forecast_bc_slim['upper_95'] = None

final_all = pd.concat([forecast_a_slim, forecast_bc_slim], ignore_index=True)
print(f"Tổng forecast toàn bộ 3,331 cặp: {len(final_all)} dòng")
print(final_all['method'].value_counts())

final_all.to_csv(f"{PROCESSED}/production_forecast_ALL.csv", index=False)
print("\n=== Đã lưu production_forecast_ALL.csv - file forecast cuối cùng cho toàn bộ dataset ===")

C:\Users\HP\AppData\Local\Temp\ipykernel_16032\2136461183.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_all = pd.concat([forecast_a_slim, forecast_bc_slim], ignore_index=True)


Tổng forecast toàn bộ 3,331 cặp: 13324 dòng
method
SARIMAX           11388
Seasonal_Naive     1452
LightGBM            484
Name: count, dtype: int64

=== Đã lưu production_forecast_ALL.csv - file forecast cuối cùng cho toàn bộ dataset ===
